In [ ]:
# Cell A: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

In [ ]:
# Cell B: Generate synthetic classification dataset
n_samples = 500

# Features
df = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.normal(50000, 15000, n_samples),
    'education_years': np.random.randint(8, 20, n_samples),
    'experience': np.random.randint(0, 40, n_samples),
    'city_type': np.random.choice(['urban', 'suburban', 'rural'], n_samples)
})

# Create target based on features (with some noise)
score = (df['income'] / 10000 + df['education_years'] * 2 + 
         df['experience'] * 0.5 + np.random.normal(0, 5, n_samples))
df['target'] = (score > score.median()).astype(int)

In [ ]:
# Cell C: Feature engineering
df['income_per_exp'] = df['income'] / (df['experience'] + 1)
df['age_education_ratio'] = df['age'] / df['education_years']
df['is_senior'] = (df['age'] > 50).astype(int)

In [ ]:
# Cell D: Encode categorical variables
le = LabelEncoder()
df['city_type_encoded'] = le.fit_transform(df['city_type'])

# One-hot encode
city_dummies = pd.get_dummies(df['city_type'], prefix='city')
df = pd.concat([df, city_dummies], axis=1)

In [ ]:
# Cell E: Prepare features and target
feature_cols = ['age', 'income', 'education_years', 'experience',
                'income_per_exp', 'age_education_ratio', 'is_senior',
                'city_urban', 'city_suburban', 'city_rural']

X = df[feature_cols].copy()
y = df['target'].copy()

In [ ]:
# Cell F: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Cell G: Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

In [ ]:
# Cell H: Train model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

In [ ]:
# Cell I: Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Cell J: Evaluate and create results
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': model.coef_[0],
    'abs_coefficient': np.abs(model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

results = {
    'train_accuracy': train_accuracy,
    'test_accuracy': test_accuracy,
    'n_train': len(X_train),
    'n_test': len(X_test),
    'n_features': len(feature_cols),
    'top_feature': feature_importance.iloc[0]['feature']
}